# Gold: hospital_daily_summary

Builds `adb_medsync.gold.hospital_daily_summary` per the Silver-to-Gold STTM: one row per hospital per day with lab test volume, abnormal/critical rates and a day-over-day trend (LAG). Driven by `silver.lab_results`, LEFT JOINed to `silver.hospitals` and `silver.calendar`. Full rebuild.

In [0]:
%sql
-- Target: adb_medsync.gold.hospital_daily_summary | Grain: one row per hospital per day | Load: FULL
CREATE OR REPLACE TABLE adb_medsync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volume, abnormal/critical rates and day-over-day trend (STTM silver-to-gold)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                              AS summary_date,
    COUNT(*)                               AS total_tests,
    COUNT(DISTINCT patient_id)             AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)              AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM adb_medsync.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  d.hospital_id                                                             AS hospital_id,
  h.hospital_name                                                           AS hospital_name,
  d.summary_date                                                            AS summary_date,
  c.day_name                                                                AS day_name,
  c.is_weekend                                                              AS is_weekend,
  CAST(d.total_tests AS INT)                                                AS total_tests,
  CAST(d.unique_patients_tested AS INT)                                     AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                        AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                        AS critical_test_count,
  CAST(d.abnormal_test_count * 100.0 / d.total_tests AS DECIMAL(5,2))       AS abnormal_rate_pct,
  CAST(d.critical_test_count * 100.0 / d.total_tests AS DECIMAL(5,2))       AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                       AS prev_day_total_tests,
  CAST(
    CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
         ELSE (d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests
    END AS DECIMAL(6,2))                                                    AS test_volume_change_pct
FROM with_lag d
LEFT JOIN adb_medsync.silver.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN adb_medsync.silver.calendar c
  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT)

In [0]:
target_table = "adb_medsync.gold.hospital_daily_summary"
updated_df = spark.read.table(target_table)
record_count = updated_df.count()

print(f"Records loaded into {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))